# Movie Analytics - Data Preparation

Load CSVs, clean, upload to MySQL, verify.

## Imports

In [ ]:
import pandas as pd
from pathlib import Path
import pymysql
from sqlalchemy import create_engine

## CSV folder path

In [ ]:
csv_folder = Path(r'C:/Users/Chandrasekar A/Documents/Python_vs/Movie_Analytics/Movie_Analytics_csv')

## List CSV files

In [ ]:
csv_files = sorted(csv_folder.glob('*.csv'))
for file in csv_files:
    print(file.name)

## Inspect files

In [ ]:
for file in csv_files:
    df = pd.read_csv(file)
    print(f"{file.name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

## Load movies

In [ ]:
movies = pd.read_csv(csv_folder / 'movies.csv')
movies.head()

In [ ]:
movies.info()

## Clean movies

In [ ]:
movies = movies.drop_duplicates()
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

numeric_cols = ['runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count']
for col in numeric_cols:
    movies[col] = pd.to_numeric(movies[col], errors='coerce')

movies['title'] = movies['title'].astype(str).str[:200]
movies['original_language'] = movies['original_language'].astype(str).str[:10]
movies['status'] = movies['status'].astype(str).str[:50]
movies['overview'] = movies['overview'].astype(str).str[:1000]

movies['release_year'] = movies['release_date'].dt.year

print(f"Cleaned. {len(movies):,} movies ready.")

## Load and clean other files

In [ ]:
genres = pd.read_csv(csv_folder / 'genres.csv')
genres = genres.drop_duplicates()

movie_genres = pd.read_csv(csv_folder / 'movie_genres.csv')
movie_genres = movie_genres.drop_duplicates()

cast = pd.read_csv(csv_folder / 'cast.csv')
cast = cast.drop_duplicates()
cast['actor_name'] = cast['actor_name'].astype(str).str[:200]
cast['character_name'] = cast['character_name'].astype(str).str[:200]

crew = pd.read_csv(csv_folder / 'crew.csv')
crew = crew.drop_duplicates()
crew['person_name'] = crew['person_name'].astype(str).str[:200]
crew['job'] = crew['job'].astype(str).str[:100]
crew['department'] = crew['department'].astype(str).str[:100]

keywords = pd.read_csv(csv_folder / 'movie_keywords.csv')
keywords = keywords.drop_duplicates()
keywords['keyword_name'] = keywords['keyword_name'].astype(str).str[:100]

print("All files loaded and cleaned.")

## Connect to MySQL

In [ ]:
MYSQL_HOST = 'localhost'
MYSQL_USER = 'root'
MYSQL_PASSWORD = '1234'
DATABASE_NAME = 'Movie analytics'

connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

## Create database

In [ ]:
cursor = connection.cursor()
cursor.execute(f"DROP DATABASE IF EXISTS `{DATABASE_NAME}`")
cursor.execute(f"CREATE DATABASE `{DATABASE_NAME}`")
cursor.execute(f"USE `{DATABASE_NAME}`")
cursor.close()

print(f"Database '{DATABASE_NAME}' created.")

## Upload to MySQL

In [ ]:
engine = create_engine(f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{DATABASE_NAME}')

movies.to_sql('movies', engine, if_exists='replace', index=False)
genres.to_sql('genres', engine, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', engine, if_exists='replace', index=False)
cast.to_sql('cast', engine, if_exists='replace', index=False)
crew.to_sql('crew', engine, if_exists='replace', index=False)
keywords.to_sql('movie_keywords', engine, if_exists='replace', index=False)

print("Data uploaded.")

## Verify row counts

In [ ]:
connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=DATABASE_NAME
)

tables = ['movies', 'genres', 'movie_genres', 'cast', 'crew', 'movie_keywords']

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", connection)
    print(f"{table}: {df['count'].iloc[0]:,} rows")

connection.close()

## Sample query - top rated movies

In [ ]:
connection = pymysql.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=DATABASE_NAME
)

query = """
SELECT title, vote_average, vote_count, release_year
FROM movies
WHERE vote_count > 1000
ORDER BY vote_average DESC
LIMIT 10
"""

top_movies = pd.read_sql(query, connection)
connection.close()

top_movies

## Done

Run the dashboard: `streamlit run app.py` → http://localhost:8501